
# Yelp Dataset - Data Engineering Assessment

## Overview
    This notebook contains an end-to-end data pipeline designed to process the Yelp dataset. The main objective is to ingest raw data, model it for the Business Intelligence (BI) team, and run a SQL query to identify "Rising Star" businesses. 

## Why is everything in a single notebook?
    In a real-world production environment, these steps are usually separated into different scripts or jobs (e.g., one for ingestion, another for transformation, and a separate environment for data analysis). However, for this assessment, I consolidated everything into a single notebook to provide a clear and concise presentation. This approach makes it easier to review the code and understand the complete data lineage—from reading the raw JSON files to the final SQL output—in one place.

## Key Considerations & Tech Stack
    * **PySpark:** Used for data ingestion and transformation because of its excellent performance and scalability with large datasets.
    * **Delta Lake:** Chosen as the storage format. It provides ACID transactions, ensures data reliability, and optimizes performance for BI workloads.
    * **Data Modeling:** I implemented a **Star Schema** following the **Medallion Architecture** (Bronze, Silver, Gold). This makes it very easy for the BI team to create reports and dashboards.

## Why are the cells separated?
    The code is divided into multiple cells to separate concerns. Each cell represents a specific stage in the data lifecycle (e.g., Bronze layer, Silver layer, Gold layer). This modular approach makes the pipeline easier to read, test, and debug.

In [0]:
from pyspark.sql.functions import col, to_date, current_date, date_sub

### 1. Bronze Layer (Raw Data Ingestion)
In this first step, I ingest the raw data from our storage. The Bronze layer keeps the data in its original format. I am reading the JSON files directly from the Databricks Volume to create my initial DataFrames.

In [0]:
# --- Path Configuration ---
base_path = "/Volumes/main/default/raw_files/"

# Reading the raw JSON files
df_biz_raw = spark.read.json(f"{base_path}yelp_academic_dataset_business.json")
df_user_raw = spark.read.json(f"{base_path}yelp_academic_dataset_user.json")
df_review_raw = spark.read.json(f"{base_path}yelp_academic_dataset_review.json")

print("Bronze Layer loaded successfully.")

Bronze Layer loaded successfully.


### 2. Bronze to Silver (Data Transformation)
Here I transform the raw data to create our Silver layer. The goal is to clean the data and select only the necessary columns for our business dimensions. I also remove duplicate records to ensure data quality.

In [0]:
# Transforming Business Data
df_biz_silver = df_biz_raw.select(
    "business_id", "name", "city", "state", "stars", "review_count", "categories"
).dropDuplicates(["business_id"])

# Transforming User Data
df_user_silver = df_user_raw.select(
    "user_id", "name", "review_count", "yelping_since", "average_stars"
).dropDuplicates(["user_id"])

print("Transformations for Silver Layer applied.")

Transformations for Silver Layer applied.


### 3. Silver Layer (Storage)
Now I save the cleaned data as Delta tables. I use the Delta format because it provides ACID transactions, scalability, and better performance. These tables will act as my dimension tables (`dim_business` and `dim_users`).

In [0]:
# Saving dimension tables in Delta format
df_biz_silver.write.format("delta").mode("overwrite").saveAsTable("dim_business")
df_user_silver.write.format("delta").mode("overwrite").saveAsTable("dim_users")

print("Silver dimension tables saved in Delta format.")

Silver dimension tables saved in Delta format.


### 4. Silver to Gold (Fact Table Preparation)
In this step, we prepare the data for the Gold layer. We transform the reviews data to create our Fact table. We format the date column to facilitate our SQL queries later, especially for the "Rising Stars" analysis.

In [0]:
# Transforming Review Data for the Fact Table
# I cast the string date to a proper DateType
df_fact_reviews = df_review_raw.select(
    "review_id", "business_id", "user_id", "stars", 
    to_date("date").alias("review_date")
)

print("Fact table transformations applied.")

Fact table transformations applied.


### 5. Gold Layer (Storage)
Finally, I save the Fact table. The Gold layer is designed for reading and reporting. This `fact_reviews` table is now ready to be consumed by the BI team and used in our SQL queries.

In [0]:
# Saving the fact table in Delta format
df_fact_reviews.write.format("delta").mode("overwrite").saveAsTable("fact_reviews")

print("Gold Fact table saved. The pipeline is complete!")

Gold Fact table saved. The pipeline is complete!


### 6. SQL Query: Identifying "Rising Stars"

In this final step, I use SQL to answer the main business question: identifying the "Rising Star" businesses. According to the requirements, a business qualifies if it meets two conditions:
1. It has received at least 10 reviews in the past year.
2. Its average rating in the past year is at least 1 star higher than its average rating before that period.

**Technical Note on the Timeline:**
Because the Yelp dataset is static, I cannot use the current system date (`current_date()`) to calculate the "past year". Instead, I created a Common Table Expression (CTE) called `GlobalAnchor` to find the most recent review date in the entire dataset. I treat this maximum date as "today". This ensures the query remains accurate and reproducible regardless of when it is executed.

In [0]:
%sql
-- Identify "Rising Stars"

WITH GlobalAnchor AS (
    -- Finding the latest date in the dataset to act as "today"
    SELECT MAX(review_date) as max_date FROM fact_reviews
),
PastYearMetrics AS (
    -- Calculating metrics for the "past year" (Criterion A: >= 10 reviews)
    SELECT 
        r.business_id,
        AVG(r.stars) as avg_rating_recent,
        COUNT(r.review_id) as count_recent
    FROM fact_reviews r
          CROSS JOIN GlobalAnchor g
    WHERE 1 = 1
            AND r.review_date >= date_sub(g.max_date, 365)
    GROUP BY r.business_id
    HAVING COUNT(r.review_id) >= 10
),
HistoricalMetrics AS (
    -- Calculating historical metrics (before the past year)
    SELECT 
        r.business_id,
        AVG(r.stars) as avg_rating_historical
    FROM fact_reviews r
        CROSS JOIN GlobalAnchor g
    WHERE 1 = 1
            AND r.review_date < date_sub(g.max_date, 365)
    GROUP BY r.business_id
)

-- Final Result: Joining CTEs and filtering by the 1-star improvement (Criterion B)
SELECT 
    b.business_id,
    b.name,
    ROUND(p.avg_rating_recent, 2) as avg_rating_past_year,
    ROUND(h.avg_rating_historical, 2) as avg_rating_before,
    p.count_recent as total_reviews_past_year
FROM PastYearMetrics p
      LEFT JOIN HistoricalMetrics h 
        ON p.business_id = h.business_id
      LEFT JOIN dim_business b 
        ON p.business_id = b.business_id
WHERE 1 = 1
        AND p.avg_rating_recent >= (h.avg_rating_historical + 1)
ORDER BY 
        (p.avg_rating_recent - h.avg_rating_historical) DESC

business_id,name,avg_rating_past_year,avg_rating_before,total_reviews_past_year
ncqjFACGWbWzDWSUyIcX_A,Southwest Blinds & Shutters,4.8,1.0,10
ith4RsN3YwQeE3WWQ3Af4w,Little Golden Goose,4.33,1.0,12
UnsaMTTvRPuy-IzOStXfGQ,Kountry Kitchen,3.94,1.0,16
CEiV5uTE84QQ4NRCJL6aTg,Touch Nail Spa II,4.38,1.5,47
Zi90uTf0F3YSCp8LjyxTRQ,Lavish Nails,4.8,2.0,10
9C7xp6BLWDO6LLYZJrBk_A,Corner Pub Cool Springs,3.56,1.0,25
VMwq77v1RxH8OHEJcY7zag,Northern Ophthalmic Assoc,4.7,2.5,10
Oe1i1zj63jCPNqt1PAyjzQ,Roto-Rooter Plumbing & Water Cleanup,4.27,2.14,22
_VGi4CJTIzUfSclxSqS4cw,Parx Beer Garden,5.0,2.92,12
lm-KtpMuHoPVCkC7zpr_xg,The Perfect Cup Cafe,5.0,2.96,17
